In [9]:
# Librerías a instalar
# !pip install deep-translator -q
# !pip install langdetect -q
# !pip install langdetect

In [10]:
# Librerías
from langdetect import detect
from langdetect.detector_factory import LangDetectException
from deep_translator import GoogleTranslator
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import time # Importamos el módulo time

## Detección del idioma y traducción al español

In [11]:
def detectar_idioma(texto):
    try:
        if pd.isna(texto) or str(texto).strip() == '': # si es un NA o sólo espacios vacíos no se traduce
            return 'es'
        return detect(str(texto)) # devuelve un string con el idioma en el que está
    except LangDetectException:
        return 'desconocido' #si no entiende el idioma devuelve desconocido

In [12]:
def traducir_lote(textos):
    try:
        return GoogleTranslator(source='auto', target='es').translate_batch(textos) #le decimos coge el idioma que sea y me lo pasas al español y translate batch es para traducir por lotes
    except Exception as e:
        print(f"Error en lote: {e}")
        return textos  # si falla, devuelve los originales

## Gestión de NaN

In [13]:
def gestionar_nulos(df, columnas):
    for col in columnas:
        df[col] = df[col].fillna('')
    df = df[~df[columnas].apply(lambda row: all(v == '' for v in row), axis=1)] # comprobación de si en una fila están todas las columnas vacías y nos quedamos con las que no son nan
    return df

## Traducción completa

In [14]:
def traduccion(columna, tamaño_lote=50, max_workers=10, checkpoint_path='checkpoint.csv', delay_between_batches=1): # detecta el idioma y si no es español lo traduce
    print("  Detectando idiomas...")
    idiomas = columna.apply(detectar_idioma)

    mask_traducir = idiomas != 'es'
    indices_traducir = columna[mask_traducir].index
    textos_traducir = columna[mask_traducir].tolist()

    print(f"  Total: {len(columna)} | En español: {(~mask_traducir).sum()} | A traducir: {mask_traducir.sum()}")

    resultado = columna.copy()

    # Cargar checkpoint si existe (por si se cayó antes)
    if os.path.exists(checkpoint_path):
        print("  Cargando checkpoint previo...")
        checkpoint = pd.read_csv(checkpoint_path, index_col=0).squeeze()
        ya_hechos = checkpoint.index
        indices_traducir = indices_traducir.difference(ya_hechos)
        textos_traducir = columna[indices_traducir].tolist()
        resultado.update(checkpoint)
        print(f"  Ya traducidos: {len(ya_hechos)} | Quedan: {len(textos_traducir)}")

    # Crear lotes
    lotes = [
        (textos_traducir[i:i+tamaño_lote], indices_traducir[i:i+tamaño_lote])
        for i in range(0, len(textos_traducir), tamaño_lote)
    ]
    total_lotes = len(lotes)
    completados = 0

    # Traducir en paralelo
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = {
            executor.submit(traducir_lote, lote): indices
            for lote, indices in lotes
        }

        for futuro in as_completed(futuros):
            indices_lote = futuros[futuro]
            try:
                traducidos = futuro.result()
                for idx, texto in zip(indices_lote, traducidos):
                    resultado[idx] = texto
            except Exception as e:
                print(f"  Error en lote: {e}")

            completados += 1

            # Guardar checkpoint cada 50 lotes
            if completados % 50 == 0:
                resultado[mask_traducir].to_csv(checkpoint_path)
                print(f"  {completados}/{total_lotes} lotes completados ({completados/total_lotes*100:.1f}%) -- Checkpoint guardado")

            # Añadir un retardo entre lotes para evitar bloqueos por rate limit
            time.sleep(delay_between_batches)

    # Guardar checkpoint final
    resultado[mask_traducir].to_csv(checkpoint_path)
    print("Traducción completada!")
    return resultado

## Ejecución

In [15]:
df_comentarios_limpio = pd.read_csv('datos/procesados/df_comentarios_limpio.csv')

df_comentarios_limpio = gestionar_nulos(df_comentarios_limpio, df_comentarios_limpio.columns.tolist())
print(df_comentarios_limpio.shape)
df_comentarios_limpio.head()

(58720, 11)


,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,comentario_media_hotel
0,meliá valencia,españa,habitación meliá con 1 cama extragrande,1,pareja,excelente,"todo la habitación súper amplia, la cama extra...",,10.0,2026-04-15,fabuloso
1,meliá valencia,españa,habitación supreme,1,persona que viaja sola,mi referente para alojarme en valencia,ya lo conocía de anteriores estancias. me enca...,,9.0,2026-04-11,fabuloso
2,meliá valencia,chile,habitación meliá con 1 cama extragrande,3,pareja,hotel funcional en lugar alejado,el lobby y cafeteria,ascensores lentos,8.0,2026-04-10,fabuloso
3,meliá valencia,españa,alojamiento the level supreme,3,familia,solo perfecto!,habitacion a la planta 24 high level - a est...,,10.0,2026-04-07,fabuloso
4,meliá valencia,perú,habitación meliá,2,familia,fantástico,"habitación cómoda, la ubicación muy buena y el...",,9.0,2026-04-01,fabuloso


In [ ]:
df_comentarios_limpio['comentario_general'] = traduccion(
    df_comentarios_limpio['comentario_general'],
    tamaño_lote=32,
    checkpoint_path='ckpt_comentario_general.csv',
    delay_between_batches=5 # Aumentamos el retardo para evitar el límite de peticiones
)

  Detectando idiomas...


In [ ]:
df_comentarios_limpio['positivo'] = traduccion(
    df_comentarios_limpio['positivo'],
    tamaño_lote=32,
    checkpoint_path='ckpt_positivo.csv',
    delay_between_batches=5 # Aumentamos el retardo
)

  Detectando idiomas...
  Total: 58720 | En español: 23973 | A traducir: 34747
  50/695 lotes completados (7.2%)
  100/695 lotes completados (14.4%)


In [ ]:
df_comentarios_limpio['negativo'] = traduccion(
    df_comentarios_limpio['negativo'],
    tamaño_lote=32,
    checkpoint_path='ckpt_negativo.csv',
    delay_between_batches=5 # Aumentamos el retardo
)

In [ ]:
df_comentarios_limpio.to_csv('comentarios_traducidos.csv', index=False)
print('Archivo guardado: comentarios_traducidos.csv')
df_comentarios_limpio.head()